# Setup & Dependencies

In [ ]:
!pip install ultralytics

In [ ]:
import os, json, shutil, random
from tqdm import tqdm
from collections import defaultdict

# Dataset Preparation (RPC to YOLO Format)

In [ ]:
RPC_PATH = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset"
WORK_DIR = "/kaggle/working/rpc"

for split in ["train", "val", "test"]:
    os.makedirs(f"{WORK_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/labels/{split}", exist_ok=True)

In [ ]:
with open(f"{RPC_PATH}/instances_val2019.json") as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]

img_map = {img["id"]: img for img in images}

ann_map = defaultdict(list)
for ann in annotations:
    ann_map[ann["image_id"]].append(ann)

In [ ]:
img_ids = list(img_map.keys())
random.shuffle(img_ids)

SPLIT = 0.5   # balanced

train_ids = set(img_ids[:int(SPLIT * len(img_ids))])
val_ids   = set(img_ids[int(SPLIT * len(img_ids)):])

## Train/Validation Split & Label Conversion

In [ ]:
def convert_and_save(img_ids, split):
    for img_id in tqdm(img_ids):
        img = img_map[img_id]
        file_name = img["file_name"]
        h, w = img["height"], img["width"]

        src = os.path.join(RPC_PATH, "val2019", file_name)
        dst = f"{WORK_DIR}/images/{split}/{file_name}"

        if not os.path.exists(src):
            continue

        shutil.copy(src, dst)

        label_path = f"{WORK_DIR}/labels/{split}/{file_name.replace('.jpg','.txt')}"

        with open(label_path, "w") as f:
            for ann in ann_map[img_id]:
                x, y, bw, bh = ann["bbox"]

                xc = (x + bw/2) / w
                yc = (y + bh/2) / h
                bw /= w
                bh /= h

                cls = ann["category_id"] - 1
                f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

convert_and_save(train_ids, "train")
convert_and_save(val_ids, "val")

## Test Set Sampling & Preparation

In [ ]:
from collections import defaultdict
import random

with open(f"{RPC_PATH}/instances_test2019.json") as f:
    test_data = json.load(f)

# ✅ Step 1: sample 5000 images
all_images = test_data["images"]
sampled_images = random.sample(all_images, 5000)

# build image dict
test_images = {img["id"]: img for img in sampled_images}
selected_ids = set(test_images.keys())

# ✅ Step 2: filter annotations for selected images
test_annotations = [
    ann for ann in test_data["annotations"]
    if ann["image_id"] in selected_ids
]

# build mapping
test_ann_map = defaultdict(list)
for ann in test_annotations:
    test_ann_map[ann["image_id"]].append(ann)

for img_id, img in tqdm(test_images.items()):
    file_name = img["file_name"]
    h, w = img["height"], img["width"]

    src = os.path.join(RPC_PATH, "test2019", file_name)
    dst = os.path.join(WORK_DIR, "images/test", file_name)

    if not os.path.exists(src):
        continue

    shutil.copy(src, dst)

    label_path = os.path.join(WORK_DIR, "labels/test", file_name.replace(".jpg",".txt"))

    with open(label_path, "w") as f:
        for ann in test_ann_map[img_id]:
            x, y, bw, bh = ann["bbox"]

            xc = (x + bw/2) / w
            yc = (y + bh/2) / h
            bw /= w
            bh /= h

            cls = ann["category_id"] - 1
            f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

## Dataset Configuration (YAML)

In [ ]:
yaml_path = f"{WORK_DIR}/rpc.yaml"

names = [str(i) for i in range(200)]

with open(yaml_path, "w") as f:
    f.write(f"""
path: {WORK_DIR}
train: images/train
val: images/val
test: images/test

nc: 200
names: {names}
""")

# Model Training (YOLOv8)

## YOLOv8 Training Strategy

The model is fine-tuned using a pretrained YOLOv8m with a **controlled learning setup** to avoid rapid overfitting and ensure stable convergence.

### Training Logic

- **Low learning rate (`lr0=3e-4`)** → slows down learning and prevents early saturation  
- **SGD optimizer** → provides stable and smoother updates  
- **Warmup (5 epochs)** → gradual start to avoid unstable gradients  
- **Backbone freezing (`freeze=10`)** → preserves pretrained features and reduces overfitting  


---

### Augmentation Strategy

Augmentations are designed to simulate **real checkout scenarios**:

#### 🔹 Lighting Variations
- `hsv_h`, `hsv_s`, `hsv_v`
- Simulates different store lighting and reflections

#### 🔹 Geometric Transformations
- `degrees`, `translate`, `scale`, `shear`
- Mimics camera angle and positioning changes

#### 🔹 Scene Complexity
- `mosaic`, `mixup`, `copy_paste`
- Creates multi-object and cluttered scenes

#### 🔹 Occlusion Handling
- `erasing`
- Simulates partially hidden products

---





In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=yaml_path,
    epochs=22,          
    imgsz=800,
    batch=16,
    optimizer="SGD",
    lr0=3e-4,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.005,

    warmup_epochs=5,

    hsv_h=0.08,
    hsv_s=0.8,
    hsv_v=0.7,

    degrees=10,
    translate=0.2,
    scale=0.5,
    shear=2,

    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,

    erasing=0.4,

    close_mosaic=10,
    freeze=10
)

## Model Evaluation (Detection Metrics)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

metrics = model.val(
    data=yaml_path,
    split="test",
    conf=0.25,
    iou=0.6
)

print("\n========= YOLO DETECTION METRICS =========")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

# Part 2: Inference Pipeline

In [ ]:
YOLO_CKPT = "runs/detect/train9/weights/best.pt" #write the path of the model wherever it is saved
# YOLO_CKPT = "/kaggle/input/models/khushalnikam/yolocheckp/pytorch/default/1/best.pt"

In [ ]:
yolo_model = YOLO(YOLO_CKPT)

In [ ]:
TEST_IMG_DIR = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/test2019"
TEST_JSON = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_test2019.json"

In [ ]:
from collections import defaultdict, Counter
import numpy as np
import json, os, cv2
from tqdm import tqdm

## IoU Calculation

In [ ]:
def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    return interArea / float(boxAArea + boxBArea - interArea)

## RPC Metrics Computation

In [ ]:
from collections import defaultdict, Counter
import numpy as np

def compute_rpc_metrics(gt_counts, pred_counts):

    categories = set(gt_counts.keys()) | set(pred_counts.keys())

    #  ACD 
    total_gt = sum(gt_counts.values())
    total_pred = sum(pred_counts.values())
    acd = abs(total_gt - total_pred)

    # mCCD
    mccd_list = []
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        if g > 0:
            mccd_list.append(abs(p - g) / g)

    mccd = np.mean(mccd_list) if mccd_list else 0

    # mCIoU 
    inter = 0
    union = 0
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        inter += min(g, p)
        union += max(g, p)

    mciou = inter / union if union > 0 else 0

    return acd, mccd, mciou

## Evaluation on Hard Images

In [ ]:
def evaluate_rpc_metrics():
    print("\n----------------- RPC METRICS (HARD IMAGES) -------------------")


    with open(TEST_JSON) as f:
        test_data = json.load(f)

    img_map = {img['id']: img for img in test_data['images']}

    ann_by_img = defaultdict(list)
    for ann in test_data['annotations']:
        ann_by_img[ann['image_id']].append(ann)

    # Filter HARD images
    hard_ids = [
        img_id for img_id, img in img_map.items()
        if img.get("level") == "hard" or img.get("difficulty") == "hard"
    ]


    yolo_acd = []
    yolo_mccd = []
    yolo_mciou = []

    total_images = 0

    for img_id in tqdm(hard_ids):

        file_name = img_map[img_id]['file_name']
        img_path = os.path.join(TEST_IMG_DIR, file_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        gt_anns = ann_by_img[img_id]

        # GT COUNT 
        gt_counts = Counter([ann['category_id'] for ann in gt_anns])


        # YOLO PREDICTION
      
        results = yolo_model(image, verbose=False)

        yolo_labels = []
        for r in results:
            if r.boxes.cls is not None:
                yolo_labels.extend(r.boxes.cls.cpu().numpy().astype(int))

        yolo_counts = Counter(yolo_labels)

        # DINO PREDICTION
        
       


        y_acd, y_mccd, y_mciou = compute_rpc_metrics(gt_counts, yolo_counts)
       
        yolo_acd.append(y_acd)
        yolo_mccd.append(y_mccd)
        yolo_mciou.append(y_mciou)

        total_images += 1


    print("\n----------------- FINAL RESULTS -------------------")

    print("\n--- YOLO ---")
    print(f"ACD   : {np.mean(yolo_acd):.2f}")
    print(f"mCCD  : {np.mean(yolo_mccd):.2f}")
    print(f"mCIoU : {100 * np.mean(yolo_mciou):.2f}%")



## Final Results

In [ ]:
if __name__ == "__main__":

    evaluate_rpc_metrics()